In [5]:
import pandas as pd
import duckdb
import os

# connect to local duckdb database file
# create the file if it doesn't exist
con = duckdb.connect("../data/isp_churn.duckdb")

print("Connected to duckdb")

Connected to duckdb


In [6]:
# load raw csv into pandas to inspect
df = pd.read_csv("../data/internet_service_churn.csv")

print(df.shape)
print(df.head())

(72274, 11)
   id  is_tv_subscriber  is_movie_package_subscriber  subscription_age  \
0  15                 1                            0             11.95   
1  18                 0                            0              8.22   
2  23                 1                            0              8.91   
3  27                 0                            0              6.87   
4  34                 0                            0              6.39   

   bill_avg  reamining_contract  service_failure_count  download_avg  \
0        25                0.14                      0           8.4   
1         0                 NaN                      0           0.0   
2        16                0.00                      0          13.7   
3        21                 NaN                      1           0.0   
4         0                 NaN                      0           0.0   

   upload_avg  download_over_limit  churn  
0         2.3                    0      0  
1         0.0         

In [8]:
# load df into duckdb as raw table
con.execute("CREATE TABLE IF NOT EXISTS raw_churn AS SELECT * FROM df")

# verify
result = con.execute("SELECT COUNT(*) FROM raw_churn").fetchone()
print(f"Rows in duckdb: {result[0]}")

Rows in duckdb: 72274


In [9]:
# run sql query directly on duckdb table
result = con.execute("""
    SELECT 
        COUNT(*) as total_rows,
        SUM(churn) as total_churned,
        ROUND(SUM(churn) * 100.0 / COUNT(*), 2) as churn_rate
    FROM raw_churn
""").fetchdf()

print(result)

   total_rows  total_churned  churn_rate
0       72274        40050.0       55.41


In [10]:
# confirm table exists in duckdb
tables = con.execute("SHOW TABLES").fetchdf()
print(tables)

        name
0  raw_churn


In [11]:
# close connection
con.close()

print("Connection closed")

Connection closed
